# Stats 161 Midterm Project

## Data import

We select **12 consecutive weeks of Polymarket data, from June 1 to August 23, 2026**, to study wallet trading behavior. All dates and timestamps use UTC.


### 1. Setup


In [ ]:
import os
from getpass import getpass

import pandas as pd
import requests

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from io import BytesIO


### 2. Dates and access settings


In [ ]:
start_date = "2026-06-01"
end_date = "2026-08-23"
dates = pd.date_range(start_date, end_date, freq="D", tz="UTC")

base_url = "https://d1yyqbluk0oz0t.cloudfront.net"
course_password = os.environ.get("POLYMARKET_PASSWORD") or getpass("Course password: ")


### 3. Wallet trades

We combine the 84 daily trade files into `wallet_trades`, keeping all wallets and all eight original variables. Each row is a recorded fill; one order may produce multiple fills.

| Variable | Description |
|---|---|
| `wallet` | Pseudonymous wallet ID, consistent across daily files. |
| `market` | Market slug; links to `markets.slug`. |
| `time` | Fill timestamp in UTC. |
| `side` | BUY or SELL for the recorded wallet. |
| `outcome` | Traded outcome, such as Yes, No, Up, Down, or a team name. |
| `price` | Execution price per share. |
| `size` | Number of shares traded. |
| `dollars` | Recorded dollar value of the fill. |


In [ ]:
session = requests.Session()
session.auth = ("polymarketdatabase", course_password)
daily_trades = []

for day in dates:
    date_string = day.strftime("%Y-%m-%d")
    url = f"{base_url}/wallet_trades/trades-{date_string}.parquet"

    response = session.get(url, timeout=120)
    response.raise_for_status()
    day_trades = pd.read_parquet(BytesIO(response.content), dtype_backend="pyarrow")
    day_trades["time"] = pd.to_datetime(day_trades["time"], utc=True)

    # Check the file date.
    assert len(day_trades) > 0, f"Empty file: {date_string}"
    assert day_trades["time"].min().date() == day.date(), f"Unexpected date: {date_string}"
    assert day_trades["time"].max().date() == day.date(), f"Unexpected date: {date_string}"

    daily_trades.append(day_trades)
    if len(daily_trades) % 28 == 0:
        print(f"Loaded {len(daily_trades)} of {len(dates)} daily files.")

wallet_trades = pd.concat(daily_trades, ignore_index=True)
session.close()
del daily_trades, day_trades, response

print(f"Wallet trades imported: {len(wallet_trades):,} rows.")


### 4. Related market data

We import market, event, and settlement records for the markets in `wallet_trades`. All original columns are retained; key variables are listed below.

| DataFrame | Content | Key variables |
|---|---|---|
| `markets` | Market questions, outcome labels, and parent events. | `slug`, `title`, `event_slug`, `condition_id`, `outcomes`, `token_ids`, `start_date`, `end_date` |
| `events` | Event titles, descriptions, and scheduled dates. | `slug`, `title`, `description`, `start_date`, `end_date` |
| `resolutions` | Available settlement records. | `market_slug`, `resolved_yes`, `price_at_close`, `resolved_at` |

**Links:** `wallet_trades.market` → `markets.slug`; `markets.event_slug` → `events.slug`; `resolutions.market_slug` → `markets.slug`.

Market prices and volumes in the metadata reflect the current stored snapshot. Interpret resolution labels using the market's actual outcome labels.


In [ ]:
database_url = URL.create(
    "postgresql+psycopg2",
    username="polymarketdatabase.merbqatlxbrcvtbnuaqe",
    password=course_password,
    host="aws-0-us-west-2.pooler.supabase.com",
    port=6543,
    database="postgres",
)

engine = create_engine(
    database_url,
    connect_args={"sslmode": "require", "connect_timeout": 20},
    execution_options={"postgresql_readonly": True},
)

market_slugs = wallet_trades["market"].dropna().unique().tolist()

with engine.connect() as connection:
    markets = pd.read_sql_query(
        text("SELECT * FROM public.markets WHERE slug = ANY(:slugs)"),
        connection,
        params={"slugs": market_slugs},
    )
    resolutions = pd.read_sql_query(
        text("SELECT * FROM public.resolutions WHERE market_slug = ANY(:slugs)"),
        connection,
        params={"slugs": market_slugs},
    )


In [ ]:
event_slugs = markets["event_slug"].dropna().unique().tolist()

with engine.connect() as connection:
    events = pd.read_sql_query(
        text("SELECT * FROM public.events WHERE slug = ANY(:slugs)"),
        connection,
        params={"slugs": event_slugs},
    )

engine.dispose()


### 5. Import check

Confirm the row and column counts, and report traded markets without matching catalog metadata.


In [ ]:
for name, data in [("wallet_trades", wallet_trades), ("markets", markets),
                   ("events", events), ("resolutions", resolutions)]:
    print(f"{name}: {len(data):,} rows, {len(data.columns)} columns")

missing_market_slugs = set(market_slugs) - set(markets["slug"])
print(f"Traded market slugs without catalog metadata: {len(missing_market_slugs):,}")
print(f"Study period: {start_date} through {end_date} (UTC)")


## Data Cleaning

### 1. Check data quality

Check missing values, trade fields, recorded amounts (tolerance: $0.01), repeated rows, and market links. Counts can overlap.


In [ ]:
missing_values = wallet_trades.isna().any(axis=1)
invalid_side = ~wallet_trades["side"].isin(["BUY", "SELL"])
invalid_price = ~wallet_trades["price"].between(0, 1)
negative_size = wallet_trades["size"] < 0
negative_dollars = wallet_trades["dollars"] < 0
amount_mismatch = (
    wallet_trades["dollars"] - wallet_trades["price"] * wallet_trades["size"]
).abs() > 0.01
repeated_rows = wallet_trades.duplicated()
missing_metadata = ~wallet_trades["market"].isin(markets["slug"])

quality_check = pd.Series({
    "Missing values in any trade column": missing_values.sum(),
    "Side outside BUY / SELL": invalid_side.sum(),
    "Price outside [0, 1]": invalid_price.sum(),
    "Negative shares": negative_size.sum(),
    "Negative dollar value": negative_dollars.sum(),
    "Dollar difference above $0.01": amount_mismatch.sum(),
    "Repeated rows after the first": repeated_rows.sum(),
    "Trades without market metadata": missing_metadata.sum(),
}, name="Rows").to_frame()

quality_check


### 2. Prepare the analysis data

Use `clean_trades` for the next analysis steps.

The checks found no missing or invalid trade values, so no rows need to be removed.

Repeated rows are kept because identical records cannot be confirmed as duplicates without a trade ID. Trades without market metadata are also kept. `has_market_metadata` indicates whether a trade's `market` matches `markets.slug`.

Large trades and low-activity wallets are retained.


In [ ]:
clean_trades = wallet_trades.copy()
clean_trades["has_market_metadata"] = ~missing_metadata

print(f"Rows retained: {len(clean_trades):,}")
print(f"Rows removed: {len(wallet_trades) - len(clean_trades):,}")


## Trader Groups: Retail, Whales, Bots and Suspected Insiders

**Question: can we split the wallets in `clean_trades` into retail traders, whales, bots and suspected insiders, and do these groups trade and earn differently?**

The steps below continue from `clean_trades`, `markets`, `events` and `resolutions` created above.

1. Build one row of behaviour features per wallet.
2. Use settled markets to measure each wallet's profit and whether it wins more often than its buy prices imply.
3. Assign each wallet to exactly one group with transparent rules.
4. Compare how the groups trade, what they trade, and who makes money.
5. Check the groups with unsupervised clustering and a supervised model.

All aggregation runs in DuckDB directly on the pandas DataFrames, so the 66.7 million rows are never copied into new pandas tables.

### 1. Setup for the analysis

In [ ]:
import json
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

SEED = 42
GROUPS = ["Retail", "Whale", "Bot", "Suspected insider"]
COLORS = {"Retail": "#8a9bb0", "Whale": "#3d6d91", "Bot": "#c0703a", "Suspected insider": "#b8312f"}

db = duckdb.connect()
db.execute("SET TimeZone = 'UTC'")
db.register("trades", clean_trades)

def q(sql):
    return db.execute(sql).df()

### 2. Market outcomes: which outcome won?

`resolutions.resolved_yes` refers to the **first** label in `markets.outcomes` (for example "Yes", "Up" or the first team). We therefore keep binary markets with two outcome labels and record the winning label explicitly. Multi-outcome markets and unresolved markets are left out of the profit analysis but stay in the behaviour features.

In [ ]:
def as_list(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    if isinstance(v, str):
        try:
            return json.loads(v)
        except ValueError:
            return None
    return None

outcome_labels = markets[["slug", "title", "event_slug", "outcomes"]].copy()
outcome_labels["outcomes"] = outcome_labels.outcomes.map(as_list)
settled = (resolutions[["market_slug", "resolved_yes", "resolved_at"]]
           .merge(outcome_labels, left_on="market_slug", right_on="slug", how="inner"))
settled = settled[settled.outcomes.map(lambda o: isinstance(o, list) and len(o) == 2)
                  & settled.resolved_yes.notna()].copy()
settled["winning_outcome"] = np.where(settled.resolved_yes.astype(bool),
                                      settled.outcomes.str[0], settled.outcomes.str[1])
settled["resolved_at"] = pd.to_datetime(settled.resolved_at, utc=True)
settled = settled[["market_slug", "winning_outcome", "resolved_at"]]
db.register("settled", settled)

traded = q("SELECT count(DISTINCT market) AS n FROM trades").n[0]
display(Markdown(f"Binary markets with a known winner: **{len(settled):,}** of **{traded:,}** traded markets."))

### 3. Topic of each market

There is no category field in the catalog, so we tag each market from its title and its event title with keyword rules. This is a rough discovery tool, not a validated classifier, but it lets us ask whether groups specialise in different topics.

In [ ]:
TOPIC_RULES = [
    ("Crypto", r"bitcoin|btc|ethereum|eth |solana|xrp|crypto|doge|up or down"),
    ("Sports", r" vs\.? | vs |nba|nfl|mlb|nhl|fifa|world cup|premier league|uefa|tennis|open|grand prix|f1|ufc|match|game \d|win on|o/u|spread"),
    ("Politics", r"election|president|trump|senate|house|governor|mayor|primary|parliament|prime minister|vote|biden|democrat|republican"),
    ("Geopolitics", r"iran|israel|ukraine|russia|china|taiwan|ceasefire|strike|war|nato|gaza|venezuela"),
    ("Economy", r"fed |interest rate|cpi|inflation|gdp|recession|unemployment|s&p|nasdaq|stock|earnings|tariff"),
    ("Culture & tech", r"movie|album|song|oscar|grammy|spotify|netflix|openai|gpt|apple|google|elon|tweet|tiktok|youtube"),
]
topic = markets[["slug", "title", "event_slug"]].merge(events[["slug", "title"]].rename(columns={"slug": "event_slug", "title": "event_title"}),
                                                        on="event_slug", how="left")
text_ = (topic.title.fillna("") + " | " + topic.event_title.fillna("")).str.lower()
topic["topic"] = "Other"
for name, pattern in reversed(TOPIC_RULES):          # earlier rules win
    topic.loc[text_.str.contains(pattern, regex=True), "topic"] = name
topic = topic[["slug", "topic"]].rename(columns={"slug": "market"})
db.register("topic", topic)
display(topic.topic.value_counts().to_frame("markets"))

### 4. Wallet behaviour features

One row per wallet, computed over the 12 weeks:

| feature | meaning | why it helps |
|---|---|---|
| `n_fills`, `n_markets`, `active_days` | how much and how widely the wallet trades | bots trade constantly and widely |
| `total_dollars`, `avg_fill_dollars`, `max_fill_dollars` | size | whales |
| `fills_per_active_day` | intensity | bots |
| `active_hours_of_day` | how many of the 24 UTC hours the wallet ever trades in | programs trade around the clock; people sleep |
| `sell_share` | share of dollars that are sells | market makers and arbitrageurs buy and sell; most retail only buys |
| `two_sided_market_share` | share of markets where the wallet traded **both** outcomes | market makers quote both sides |
| `median_gap_seconds` | median time between consecutive fills | sub-second gaps suggest automation |
| `avg_buy_price` | dollar-weighted price paid when buying | favourites (near 1) vs long shots (near 0) |
| `concentration` | share of the wallet's dollars in its single biggest market | all-in bettors |
| `max_market_share` | wallet's largest share of one market's dollars in the sample | did it move the market? |
| `lifetime_days` | first to last fill in the sample | fresh, short-lived wallets |

In [ ]:
wallets = q('''
    WITH t AS (
        SELECT *, lag(time) OVER (PARTITION BY wallet ORDER BY time) AS prev_time FROM trades
    ),
    base AS (
        SELECT wallet,
               count(*)                                         AS n_fills,
               count(DISTINCT market)                           AS n_markets,
               count(DISTINCT CAST(time AS DATE))               AS active_days,
               count(DISTINCT extract(hour FROM time))          AS active_hours_of_day,
               sum(dollars)                                     AS total_dollars,
               avg(dollars)                                     AS avg_fill_dollars,
               max(dollars)                                     AS max_fill_dollars,
               sum(dollars) FILTER (WHERE side = 'SELL') / nullif(sum(dollars), 0) AS sell_share,
               sum(price * dollars) FILTER (WHERE side = 'BUY') /
                   nullif(sum(dollars) FILTER (WHERE side = 'BUY'), 0)            AS avg_buy_price,
               median(epoch(time) - epoch(prev_time))           AS median_gap_seconds,
               min(time) AS first_fill, max(time) AS last_fill
        FROM t GROUP BY wallet
    ),
    per_market AS (
        SELECT wallet, market, sum(dollars) AS d, count(DISTINCT outcome) AS n_outcomes
        FROM trades GROUP BY wallet, market
    ),
    market_total AS (SELECT market, sum(dollars) AS md FROM trades GROUP BY market),
    focus AS (
        SELECT p.wallet,
               max(p.d) / sum(p.d)                       AS concentration,
               max(p.d / m.md)                           AS max_market_share,
               avg((p.n_outcomes > 1)::INT)              AS two_sided_market_share
        FROM per_market p JOIN market_total m USING (market) GROUP BY p.wallet
    )
    SELECT b.*, f.concentration, f.max_market_share, f.two_sided_market_share,
           b.n_fills / b.active_days AS fills_per_active_day,
           (epoch(b.last_fill) - epoch(b.first_fill)) / 86400 AS lifetime_days
    FROM base b JOIN focus f USING (wallet)
''').set_index("wallet")
wallets["median_gap_seconds"] = wallets.median_gap_seconds.fillna(np.inf)
display(Markdown(f"**{len(wallets):,} wallets**"))
display(wallets.drop(columns=["first_fill", "last_fill"]).describe(percentiles=[0.5, 0.9, 0.99]).T.round(3))

### 5. Profit and "skill" on settled markets

For every wallet × market × outcome we net the shares bought and sold. If the wallet still holds shares at settlement, each share pays $1 when its outcome wins and $0 otherwise:

$$\text{profit} = (\text{shares bought} - \text{shares sold}) \times \text{payoff} - \text{dollars bought} + \text{dollars sold}$$

Profit alone rewards size and luck. To measure **information**, we compare each wallet's wins with what its buy prices imply. A position bought at an average price $p$ wins with probability $p$ if the market is fair, so across a wallet's positions

$$z = \frac{\text{wins} - \sum p_i}{\sqrt{\sum p_i(1-p_i)}}$$

A large positive $z$ means the wallet wins far more often than the prices it paid suggest. That is what informed (or insider) trading should look like, and what pure luck rarely produces across many positions.

In [ ]:
positions = q('''
    SELECT t.wallet, t.market, t.outcome,
           sum(size) FILTER (WHERE side = 'BUY')     AS sh_bought,
           sum(size) FILTER (WHERE side = 'SELL')    AS sh_sold,
           sum(dollars) FILTER (WHERE side = 'BUY')  AS d_bought,
           sum(dollars) FILTER (WHERE side = 'SELL') AS d_sold,
           sum(dollars) FILTER (WHERE side = 'BUY') / nullif(sum(size) FILTER (WHERE side = 'BUY'), 0) AS avg_buy_price,
           median(epoch(s.resolved_at) - epoch(t.time)) FILTER (WHERE side = 'BUY') / 3600 AS median_hours_before_resolution,
           any_value((t.outcome = s.winning_outcome)::INT) AS won
    FROM trades t JOIN settled s ON s.market_slug = t.market
    WHERE t.time <= s.resolved_at
    GROUP BY t.wallet, t.market, t.outcome
''').fillna({"sh_bought": 0, "sh_sold": 0, "d_bought": 0, "d_sold": 0})
positions["profit"] = (positions.sh_bought - positions.sh_sold) * positions.won - positions.d_bought + positions.d_sold

# Skill uses positions the wallet bought and still held (net long), at a usable price.
held = positions[(positions.sh_bought > positions.sh_sold) & positions.avg_buy_price.between(0.01, 0.99)].copy()
held["var"] = held.avg_buy_price * (1 - held.avg_buy_price)
skill = held.groupby("wallet").agg(n_resolved_positions=("won", "size"), wins=("won", "sum"),
                                   expected_wins=("avg_buy_price", "sum"), var=("var", "sum"),
                                   median_hours_before_resolution=("median_hours_before_resolution", "median"))
skill["skill_z"] = (skill.wins - skill.expected_wins) / np.sqrt(skill["var"])
pnl = positions.groupby("wallet").agg(profit=("profit", "sum"), settled_dollars_bought=("d_bought", "sum"))
pnl["roi"] = pnl.profit / pnl.settled_dollars_bought.replace(0, np.nan)

wallets = wallets.join(pnl).join(skill.drop(columns="var"))
display(Markdown(f"Wallet × market × outcome positions on settled binary markets: **{len(positions):,}**; "
                 f"wallets with at least one held position: **{len(skill):,}**."))
display(wallets[["profit", "roi", "n_resolved_positions", "skill_z", "median_hours_before_resolution"]]
        .describe(percentiles=[0.01, 0.5, 0.99]).T.round(3))

**Sanity check.** If prices are fair on average, `skill_z` across all wallets should centre near 0 with spread near 1. A strong overall shift would mean the price-implied probabilities are biased (for example the favourite–long-shot bias), which we would need to correct for before calling any wallet "skilled".

In [ ]:
z = wallets.skill_z.dropna()
z5 = wallets.loc[wallets.n_resolved_positions >= 5, "skill_z"].dropna()
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(z.clip(-6, 8), bins=80, density=True, alpha=0.5, color="#8a9bb0", label=f"all wallets (n={len(z):,})")
ax.hist(z5.clip(-6, 8), bins=80, density=True, histtype="step", lw=2, color="#3d6d91", label=f"≥5 positions (n={len(z5):,})")
xs = np.linspace(-6, 8, 300)
ax.plot(xs, stats.norm.pdf(xs), color="black", ls=":", label="N(0, 1) if prices were fair")
ax.set(xlabel="skill z (clipped to [-6, 8])", ylabel="density", title="Do wallets win more than their buy prices imply?")
ax.legend(); plt.show()
display(pd.Series({"mean z (≥5 positions)": z5.mean(), "sd z (≥5 positions)": z5.std(),
                   "share with z ≥ 3 (≥5 positions)": (z5 >= 3).mean()}).round(3).to_frame("value"))

### 6. Assigning the four groups

Rules are applied **in this order**, so each wallet gets exactly one label:

1. **Bot**: `n_markets ≥ 500` or `n_fills ≥ 2,000` (the course demo's rule), **or** automated-looking behaviour: trades in at least 20 of the 24 UTC hours *and* more than 100 fills per active day. People sleep; programs do not.
2. **Suspected insider** (non-bots only): wins far more than prices imply on a meaningful amount of money, with the footprint the course's confirmed insiders had (median wallet age 11 days, 42% of money in one market, buy price around 0.66):
   * `skill_z ≥ 3` and at least 3 settled positions and at least $1,000 bought in settled markets, **and**
   * a focused or fresh footprint: `concentration ≥ 0.4` **or** `lifetime_days ≤ 30`.
3. **Whale**: among the rest, the top **1%** by `total_dollars`.
4. **Retail**: everyone else.

"Suspected insider" is a **statistical flag**, not an accusation: it marks wallets whose wins are hard to explain by luck. There are no confirmed labels in this data, because the course's confirmed insiders use a different pseudonym scheme. Section 6.1 shows how the counts move with the thresholds.

In [ ]:
RULES = dict(bot_markets=500, bot_fills=2000, bot_hours=20, bot_fills_per_day=100,
             ins_z=3.0, ins_positions=3, ins_dollars=1000, ins_conc=0.4, ins_life=30, whale_top=0.01)

def assign_groups(w, r=RULES):
    is_bot = ((w.n_markets >= r["bot_markets"]) | (w.n_fills >= r["bot_fills"])
              | ((w.active_hours_of_day >= r["bot_hours"]) & (w.fills_per_active_day > r["bot_fills_per_day"])))
    is_ins = (~is_bot & (w.skill_z >= r["ins_z"]) & (w.n_resolved_positions >= r["ins_positions"])
              & (w.settled_dollars_bought >= r["ins_dollars"])
              & ((w.concentration >= r["ins_conc"]) | (w.lifetime_days <= r["ins_life"])))
    rest = ~is_bot & ~is_ins
    cut = w.loc[rest, "total_dollars"].quantile(1 - r["whale_top"])
    g = pd.Series("Retail", index=w.index)
    g[rest & (w.total_dollars >= cut)] = "Whale"
    g[is_ins] = "Suspected insider"
    g[is_bot] = "Bot"
    return pd.Categorical(g, categories=GROUPS), cut

wallets["group"], WHALE_CUT = assign_groups(wallets)
summary = wallets.groupby("group", observed=False).agg(wallets=("n_fills", "size"), fills=("n_fills", "sum"),
                                                       dollars=("total_dollars", "sum"), profit=("profit", "sum"))
for c in ["wallets", "fills", "dollars"]:
    summary[f"share_of_{c}"] = summary[c] / summary[c].sum()
display(Markdown(f"Whale cut-off: at least **${WHALE_CUT:,.0f}** traded in the 12 weeks."))
display(summary.style.format({"dollars": "${:,.0f}", "profit": "${:,.0f}", "share_of_wallets": "{:.2%}",
                              "share_of_fills": "{:.1%}", "share_of_dollars": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(9, 3.2))
left = np.zeros(3)
for g in GROUPS:
    vals = summary.loc[g, ["share_of_wallets", "share_of_fills", "share_of_dollars"]].to_numpy(float)
    ax.barh(["wallets", "fills", "dollars"], vals, left=left, color=COLORS[g], label=g)
    left += vals
ax.set(xlim=(0, 1), xlabel="share", title="Each group's share of wallets, fills and dollars")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.25)); plt.show()

#### 6.1 How sensitive are the groups to the thresholds?

In [ ]:
variants = {
    "baseline": {},
    "whale top 0.5%": {"whale_top": 0.005}, "whale top 5%": {"whale_top": 0.05},
    "insider z ≥ 2": {"ins_z": 2.0}, "insider z ≥ 4": {"ins_z": 4.0},
    "insider ≥ $10k": {"ins_dollars": 10000},
    "bot ≥ 1,000 fills": {"bot_fills": 1000}, "no hour-of-day bot rule": {"bot_hours": 25},
}
rows = []
for name, change in variants.items():
    g, _ = assign_groups(wallets, {**RULES, **change})
    counts = pd.Series(g).value_counts().reindex(GROUPS)
    rows.append({"variant": name, **counts.to_dict()})
display(pd.DataFrame(rows).set_index("variant"))

### 7. How do the groups trade?

Medians are used because every feature is heavily skewed. The heat map shows each group's median relative to the median across groups (10× = ten times the cross-group median).

In [ ]:
PROFILE = ["n_fills", "n_markets", "active_days", "fills_per_active_day", "active_hours_of_day",
           "median_gap_seconds", "total_dollars", "avg_fill_dollars", "max_fill_dollars", "sell_share",
           "two_sided_market_share", "avg_buy_price", "concentration", "max_market_share", "lifetime_days"]
profile = wallets.replace(np.inf, np.nan).groupby("group", observed=False)[PROFILE].median().T
display(profile.style.format("{:,.3f}"))

rel = profile.div(profile.median(axis=1), axis=0).replace(0, np.nan)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(np.log10(rel.astype(float)), cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(rel.columns)), rel.columns, rotation=20); ax.set_yticks(range(len(rel.index)), rel.index)
for i in range(rel.shape[0]):
    for j in range(rel.shape[1]):
        v = rel.iat[i, j]
        ax.text(j, i, "" if pd.isna(v) else f"{v:.2g}×", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="log10(group median / cross-group median)")
ax.set_title("Group medians relative to the cross-group median"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
panels = [("avg_fill_dollars", True), ("fills_per_active_day", True), ("active_hours_of_day", False),
          ("avg_buy_price", False), ("concentration", False), ("two_sided_market_share", False)]
for ax, (col, logy) in zip(axes.flat, panels):
    data = [wallets.loc[wallets.group == g, col].dropna().to_numpy() for g in GROUPS]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.6)
    for patch, g in zip(bp["boxes"], GROUPS):
        patch.set_facecolor(COLORS[g]); patch.set_alpha(0.75)
    ax.set_xticks(range(1, 5), ["Retail", "Whale", "Bot", "Insider?"])
    if logy: ax.set_yscale("log")
    ax.set_title(col)
plt.suptitle("Trading behaviour by group (boxes = interquartile range, outliers hidden)", y=1.01)
plt.tight_layout(); plt.show()

#### 7.1 When in the day does each group trade?

Share of each group's fills by UTC hour. A flat line means round-the-clock trading; a pronounced daily cycle means human schedules (US evenings are roughly 00–05 UTC).

In [ ]:
db.register("wallet_group", wallets[["group"]].astype(str).reset_index())
hourly = q('''
    SELECT g."group", extract(hour FROM t.time) AS hour_utc, count(*) AS fills
    FROM trades t JOIN wallet_group g USING (wallet)
    GROUP BY 1, 2
''')
hourly["share"] = hourly.fills / hourly.groupby("group").fills.transform("sum")
fig, ax = plt.subplots(figsize=(10, 3.8))
for g in GROUPS:
    h = hourly[hourly.group == g].sort_values("hour_utc")
    ax.plot(h.hour_utc, h.share, marker="o", ms=3, color=COLORS[g], label=g)
ax.axhline(1 / 24, color="grey", ls=":", label="flat (1/24)")
ax.set(xlabel="hour of day (UTC)", ylabel="share of the group's fills", title="Daily trading cycle by group")
ax.legend(ncol=5, fontsize=8); plt.show()

#### 7.2 What do the groups trade?

Share of each group's dollars by market topic (Section 3). Do whales and suspected insiders concentrate in particular topics?

In [ ]:
by_topic = q('''
    SELECT g."group", coalesce(tp.topic, 'No metadata') AS topic, sum(t.dollars) AS dollars
    FROM trades t JOIN wallet_group g USING (wallet) LEFT JOIN topic tp USING (market)
    GROUP BY 1, 2
''')
mix = by_topic.pivot_table(index="group", columns="topic", values="dollars", aggfunc="sum").reindex(GROUPS)
mix = mix.div(mix.sum(axis=1), axis=0)
display(mix.style.format("{:.1%}").background_gradient(axis=1, cmap="Blues"))
mix.plot.barh(stacked=True, figsize=(10, 3.5), colormap="tab20", title="Share of each group's dollars by topic")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8); plt.xlabel("share of dollars"); plt.show()

#### 7.3 How close to settlement do the groups buy?

Median hours between a wallet's buys and the market's settlement, on settled markets. Informed traders should buy **late but before** the price fully reflects the news, i.e. shortly before settlement at prices that are still well below 1.

In [ ]:
pos_g = positions.merge(wallets[["group"]], left_on="wallet", right_index=True)
pos_g = pos_g[pos_g.d_bought > 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for g in GROUPS:
    s = pos_g.loc[pos_g.group == g]
    axes[0].hist(np.log10(s.median_hours_before_resolution.clip(lower=0.01)), bins=50, density=True,
                 histtype="step", lw=2, color=COLORS[g], label=g)
    axes[1].hist(s.avg_buy_price.dropna(), bins=np.linspace(0, 1, 26), density=True,
                 histtype="step", lw=2, color=COLORS[g], label=g)
axes[0].set(xlabel="log10(hours before settlement)", ylabel="density", title="Timing of buys before settlement")
axes[1].set(xlabel="average buy price of the position", ylabel="density", title="Price paid on settled positions")
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

late = pos_g.assign(late_mid_price=(pos_g.median_hours_before_resolution <= 24) & pos_g.avg_buy_price.between(0.2, 0.8))
display(late.groupby("group", observed=False).agg(
    positions=("won", "size"),
    share_bought_within_24h=("median_hours_before_resolution", lambda h: (h <= 24).mean()),
    share_late_mid_price=("late_mid_price", "mean"),
    win_rate_late_mid_price=("won", lambda w: w[late.loc[w.index, "late_mid_price"]].mean())).round(3))

### 8. Who makes money?

Prediction markets are zero-sum before fees: every dollar one wallet wins on a settled market, another wallet loses. Summing profit by group shows **which groups take money from which**.

In [ ]:
money = wallets.groupby("group", observed=False).agg(
    wallets_with_settled=("profit", "count"), total_profit=("profit", "sum"),
    median_profit=("profit", "median"), median_roi=("roi", "median"),
    share_profitable=("profit", lambda p: (p.dropna() > 0).mean()),
    median_skill_z=("skill_z", "median"))
display(money.style.format({"total_profit": "${:,.0f}", "median_profit": "${:,.2f}", "median_roi": "{:+.1%}",
                            "share_profitable": "{:.1%}", "median_skill_z": "{:+.2f}"}))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(GROUPS, money.total_profit.reindex(GROUPS) / 1e6, color=[COLORS[g] for g in GROUPS])
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="total profit on settled markets ($ millions)", title="Net transfer between groups")
plt.show()

### 9. Statistical tests

Kruskal–Wallis asks whether at least one group differs on a feature. Cliff's δ (between −1 and 1) measures how different two groups are regardless of sample size; |δ| > 0.47 is conventionally large. The most informative column is **suspected insider vs whale**: both bet large amounts, so features that separate them describe *informed* money rather than *big* money.

In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return np.nan
    r = stats.rankdata(np.concatenate([a, b]))
    u = r[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    return 2 * u / (len(a) * len(b)) - 1

TEST = ["avg_fill_dollars", "fills_per_active_day", "active_hours_of_day", "sell_share", "two_sided_market_share",
        "avg_buy_price", "concentration", "max_market_share", "lifetime_days", "median_hours_before_resolution"]
rows = []
for col in TEST:
    s = {g: wallets.loc[wallets.group == g, col].replace(np.inf, np.nan).dropna() for g in GROUPS}
    present = [v for v in s.values() if len(v) > 0]
    h, p = stats.kruskal(*present) if len(present) > 1 else (np.nan, np.nan)
    rows.append({"feature": col, "kruskal_H": h, "p_value": p,
                 "δ insider vs retail": cliffs_delta(s["Suspected insider"], s["Retail"]),
                 "δ insider vs whale": cliffs_delta(s["Suspected insider"], s["Whale"]),
                 "δ bot vs whale": cliffs_delta(s["Bot"], s["Whale"]),
                 "δ whale vs retail": cliffs_delta(s["Whale"], s["Retail"])})
tests = pd.DataFrame(rows).set_index("feature")
display(tests.style.format({"kruskal_H": "{:,.0f}", "p_value": "{:.1e}"}, precision=2)
        .background_gradient(subset=[c for c in tests.columns if c.startswith("δ")], cmap="RdBu_r", vmin=-1, vmax=1))

### 10. Does clustering find the same groups without the rules?

We standardize log-transformed behaviour features (no profit or skill, so the clustering cannot "see" the insider rule) and run k-means with k = 4 on a random sample of wallets. If the rule-based groups are natural behavioural types, the clusters should line up with them (adjusted Rand index near 1).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.decomposition import PCA

BEHAVIOUR = ["n_fills", "n_markets", "active_days", "fills_per_active_day", "active_hours_of_day",
             "avg_fill_dollars", "total_dollars", "sell_share", "two_sided_market_share", "avg_buy_price",
             "concentration", "max_market_share", "lifetime_days"]
LOGS = ["n_fills", "n_markets", "active_days", "fills_per_active_day", "avg_fill_dollars", "total_dollars", "lifetime_days"]

def behaviour_matrix(w):
    X = w[BEHAVIOUR].copy()
    X[LOGS] = np.log1p(X[LOGS])
    X["max_market_share"] = np.log10(X.max_market_share + 1e-6)
    return X.fillna(X.median())

# Keep every bot, whale and suspected insider; sample retail so the fit is fast and the small groups are visible.
rng = np.random.default_rng(SEED)
retail_idx = wallets.index[wallets.group == "Retail"]
keep = wallets.index[wallets.group != "Retail"].append(
    pd.Index(rng.choice(retail_idx, size=min(50000, len(retail_idx)), replace=False)))
S = wallets.loc[keep]
Z = StandardScaler().fit_transform(behaviour_matrix(S))
km = KMeans(4, n_init=10, random_state=SEED).fit(Z)
S = S.assign(cluster=km.labels_)
display(Markdown(f"k = 4 on {len(S):,} wallets. Adjusted Rand index vs rule-based groups: "
                 f"**{adjusted_rand_score(S.group.astype(str), S.cluster):.3f}**; silhouette "
                 f"**{silhouette_score(Z, km.labels_, sample_size=min(10000, len(Z)), random_state=SEED):.3f}**."))
ct = pd.crosstab(S.cluster, S.group).reindex(columns=GROUPS)
display((ct / ct.sum()).style.format("{:.1%}"))

pc = PCA(2, random_state=SEED).fit_transform(Z)
fig, ax = plt.subplots(figsize=(8, 6))
for g in GROUPS:
    m = (S.group == g).to_numpy()
    ax.scatter(pc[m, 0], pc[m, 1], s=3 if g == "Retail" else 14, alpha=0.2 if g == "Retail" else 0.7,
               color=COLORS[g], label=g)
ax.set(xlabel="PC1", ylabel="PC2", title="Wallets in behaviour space (retail sampled)"); ax.legend(markerscale=3)
plt.show()

### 11. Can behaviour alone predict who is a whale, a bot or a suspected insider?

A gradient-boosted classifier predicts the group from behaviour features only. Profit and `skill_z` are **excluded**, because the insider rule uses them. If the model still finds suspected insiders, there is a behavioural signature beyond "they won". Scores come from 5-fold cross-validation on the same sample as Section 10.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# Groups with fewer than 5 wallets cannot be cross-validated; leave them out of this model.
counts = S.group.astype(str).value_counts()
S_cls = S[S.group.astype(str).isin(counts[counts >= 5].index)]
Xs = behaviour_matrix(S_cls)
ys = S_cls.group.astype(str).to_numpy()
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, class_weight="balanced", random_state=SEED)
pred = cross_val_predict(clf, Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
print(classification_report(ys, pred, labels=GROUPS, digits=3, zero_division=0))
display(pd.DataFrame(confusion_matrix(ys, pred, labels=GROUPS), index=[f"true {g}" for g in GROUPS],
                     columns=[f"pred {g}" for g in GROUPS]))

clf.fit(Xs, ys)
imp = permutation_importance(clf, Xs, ys, n_repeats=5, random_state=SEED, scoring="balanced_accuracy")
pd.Series(imp.importances_mean, index=BEHAVIOUR).sort_values().plot.barh(
    figsize=(8, 4.5), color="#3d6d91", title="Which behaviour features identify the groups? (permutation importance)")
plt.xlabel("drop in balanced accuracy when the feature is shuffled"); plt.show()

### 12. Most extreme suspected insiders

The flagged wallets with the highest `skill_z`, for a closer look at their individual trades. These are pseudonymous wallets and statistical flags only.

In [ ]:
top_flags = (wallets[wallets.group == "Suspected insider"]
             .sort_values("skill_z", ascending=False)
             .head(15)[["skill_z", "wins", "n_resolved_positions", "expected_wins", "profit", "roi",
                        "settled_dollars_bought", "avg_buy_price", "concentration", "lifetime_days",
                        "median_hours_before_resolution"]])
display(top_flags.round(2))

if len(top_flags):
    example = top_flags.index[0]
    display(Markdown(f"Settled positions of the top flagged wallet `{example}`:"))
    display(positions[positions.wallet == example]
            .merge(markets[["slug", "title"]], left_on="market", right_on="slug", how="left")
            [["title", "outcome", "avg_buy_price", "d_bought", "won", "profit", "median_hours_before_resolution"]]
            .sort_values("d_bought", ascending=False).head(10).round(3))

## Limits and possible extensions

**Limits**

* **No ground truth for insiders.** "Suspected insider" means "wins much more than prices imply, with a focused or fresh footprint". Some flagged wallets will be skilled or lucky traders; some real insiders (one small bet, never seen again) cannot be detected with only a few positions.
* **12 weeks only.** Wallets that started before June 1 or continued after August 23 are cut off, so `lifetime_days` and profit are partial.
* **Repeated rows are kept** (as decided in Data Cleaning). If they turn out to be true duplicates, fills and dollars are inflated for the affected wallets.
* **Profit assumes holding to settlement.** Positions closed by selling are counted through `d_sold`; fees and rewards are not modelled.
* **Thresholds are choices.** Section 6.1 shows how the group sizes respond.

**Extensions worth considering for the final report**

1. **Event-time study for suspected insiders.** For each flagged wallet's winning market, plot the market price around the wallet's buy time (from the order-book files or from all wallets' fills). Insiders should buy *before* the price jump; skilled traders react *after* news.
2. **Correct `skill_z` for the favourite–long-shot bias.** Fit a calibration curve (price → actual win rate) on all settled markets and use the calibrated probability instead of the raw price as the expected win rate.
3. **Copy-trading test.** If whales or suspected insiders are informed, does following their trades predict the next-hour price move? This turns the grouping into a forecasting model.
4. **Wallet networks.** Wallets that repeatedly trade the same markets within seconds of each other may be one person or a coordinated group; a co-trading graph and community detection could reveal clusters of linked wallets.
5. **Bot sub-types.** Separate market makers (two-sided, balanced buy/sell, flat daily cycle) from arbitrageurs (fast, both outcomes across related markets) and directional bots.
6. **Retail learning.** Do retail wallets that lose early stop trading, and do the survivors get better over time?

In [ ]:
db.close()